# QLoRA Finetuning for News Summarization

This notebook demonstrates how to fine-tune a large language model (like Llama-3-8B) to generate highly structured financial news summaries using **QLoRA (Quantized Low-Rank Adaptation)**.

Instead of fine-tuning the entire model (which requires massive compute), QLoRA first quantizes the base model to 4-bit precision to save memory, then freezes the pre-trained model weights and injects trainable rank decomposition matrices into each layer of the Transformer architecture. This allows us to fine-tune a massive model on a single consumer GPU.

In [ ]:
# Install required training libraries if not already installed
!pip install -q peft transformers datasets trl bitsandbytes

## 1. Load Dataset
We will use a standard news summarization dataset. In a real financial application, you would replace this with a dataset of raw articles paired with your preferred analyst summaries.

In [ ]:
from datasets import load_dataset

# Load a subset of the CNN/DailyMail summarization dataset
dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:1000]")

def format_instruction(sample):
    return f"""### Instruction:
Summarize the following news article into key bullet points.

### Input:
{sample['article']}

### Response:
{sample['highlights']}"""

# Add formatted prompt column
dataset = dataset.map(lambda x: {"text": format_instruction(x)})

## 2. Load Model in 4-bit Precision (QLoRA Step 1)
To fit the model on a standard consumer GPU, we quantize it to 4-bit precision using `bitsandbytes`. This is the 'Q' in QLoRA.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

## 3. Apply LoRA Configuration (QLoRA Step 2)
We use the `peft` library to inject the LoRA adapters into our 4-bit quantized base model.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for training
model = prepare_model_for_kbit_training(model)

# Define LoRA config
lora_config = LoraConfig(
    r=16, # Rank of the adapter
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Training Loop
We use `SFTTrainer` (Supervised Fine-Tuning) from the `trl` library to execute the training loop.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./news_summarizer_lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100, # Set to a low number for demonstration
    fp16=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

# Start training (Uncomment to run if you have a GPU)
# trainer.train()

## 5. Save the Fine-Tuned Adapters
Once training is complete, we save the lightweight LoRA weights. During inference, these weights are loaded on top of the base Llama-3-8B model.

In [ ]:
# Save the LoRA adapters
# trainer.model.save_pretrained("news-summarizer-lora-adapter")
# tokenizer.save_pretrained("news-summarizer-lora-adapter")
print("Training pipeline setup complete!")